# 05 · Style Yes, Facts No (the honest boundary)

What LoRA **cannot** do. A style/format/sentiment change is a *low-rank* adjustment,
so LoRA learns it cleanly. Injecting a body of new **facts** is not a low-rank
change — the associations are arbitrary and high-rank — so a small adapter picks up
the *phrasing* of the facts but not the facts. The right tool for changing facts is
retrieval (`rag_101`). We save the real text corpora, then measure the boundary on
the transparent toy where 'low-rank' vs 'full-rank' is exact.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print('repo root:', _p)

repo root: /workspaces/lora_101


In [2]:
from src.tasks.small_lm import style_task, facts_task, STYLE_EXAMPLES, FACT_EXAMPLES
from src.results import save_json

style, facts = style_task(), facts_task()
corpora = save_json({
    'style_task': {'name': style.name, 'kind': style.kind,
                   'train_texts': style.train_texts, 'probe': style.probe},
    'facts_task': {'name': facts.name, 'kind': facts.kind,
                   'train_texts': facts.train_texts, 'probe': facts.probe},
}, '05_task_corpora')
print('saved corpora:', corpora.name)
print('style example:', STYLE_EXAMPLES[0])
print('fact  example:', FACT_EXAMPLES[0])

saved corpora: 05_task_corpora.json
style example: Review: the plot dragged. Sentiment: negative.
fact  example: The capital of Zantovia is Mirelle.


## Measure the boundary on the transparent toy
Give LoRA a fixed small rank. The 'style' change is genuinely low-rank (LoRA fits
it); the 'facts' change is (almost) full-rank (LoRA cannot absorb it).

In [3]:
import torch
from src.config import get_settings
from src.tasks.toy import make_toy_task, make_base_model
from src.evaluation.comparison import three_way_race
from src.analysis.subspace import captured_energy

s = get_settings()
d = s.toy.dim
adapter_r = s.toy.true_rank  # a small, fixed adapter budget
style_toy = make_toy_task(dim=d, out_dim=d, true_rank=adapter_r,
                          n_samples=s.toy.n_samples, seed=s.seed)
facts_toy = make_toy_task(dim=d, out_dim=d, true_rank=d,
                          n_samples=s.toy.n_samples, seed=s.seed + 1)

def fit_quality(tk):
    base = make_base_model(tk)
    res = three_way_race(base, tk, r=adapter_r, alpha=s.lora.alpha, target_modules=['linear'],
                         steps=s.train.steps, lr=s.train.lr, seed=s.seed)
    return res['frozen'].final_loss, res['lora'].final_loss

fz_s, lo_s = fit_quality(style_toy)
fz_f, lo_f = fit_quality(facts_toy)
style_fit, facts_fit = 1 - lo_s / fz_s, 1 - lo_f / fz_f
style_cap = captured_energy(style_toy.delta, adapter_r)
facts_cap = captured_energy(facts_toy.delta, adapter_r)
print(f'STYLE (low-rank):  captures {style_cap:.2f} of update, fit quality {style_fit:.2f}')
print(f'FACTS (full-rank): captures {facts_cap:.2f} of update, fit quality {facts_fit:.2f}')

STYLE (low-rank):  captures 1.00 of update, fit quality 1.00
FACTS (full-rank): captures 0.17 of update, fit quality 0.21


## Save the contrast (PNG + JSON)

In [4]:
import matplotlib.pyplot as plt
from src.visualisation import save_figure

labels = ['style\n(low-rank)', 'facts\n(full-rank)']
fits, caps = [style_fit, facts_fit], [style_cap, facts_cap]
xpos = range(len(labels))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([p - 0.2 for p in xpos], fits, width=0.4, label='LoRA fit quality')
ax.bar([p + 0.2 for p in xpos], caps, width=0.4, label='subspace captured')
ax.set_xticks(list(xpos)); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05); ax.set_ylabel('fraction'); ax.legend()
ax.set_title(f'LoRA (rank {adapter_r}) learns low-rank style, not full-rank facts')
png = save_figure(fig, '05_style_vs_facts')
json_path = save_json({
    'adapter_rank': adapter_r, 'dim': d,
    'style': {'update': 'low-rank', 'true_rank': adapter_r, 'captured': style_cap, 'fit_quality': style_fit},
    'facts': {'update': 'full-rank', 'true_rank': d, 'captured': facts_cap, 'fit_quality': facts_fit},
}, '05_style_yes_facts_no')
print('saved:', png.name, 'and', json_path.name)

saved: 05_style_vs_facts.png and 05_style_yes_facts_no.json


## Takeaway
A low-rank adapter captures almost all of a low-rank (style) change and little of a
full-rank (facts) change — the same reason a real LoRA picks up a tone or format but
gets invented facts wrong. Change behavior with LoRA; change facts with retrieval
(`rag_101`).